# Appendix A Companion Notebook: Google Colaboratory (Colab)

**Book:** *Business Analytics and Artificial Intelligence: An Advanced Guide to Data-Driven Decision Making*  
**Book authors:** Hyunhwan "Aiden" Lee and Reo Song  
**Notebook author:** Hyunhwan Aiden Lee  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indy16mm/business-analytics-ai/blob/main/appendices/Appendix_A_Google_Colab.ipynb)

This notebook accompanies Appendix A of the book.

**License:** Use of this notebook is governed by the repository's
[Limited Companion Materials License](../LICENSE).




This notebook turns Appendix A into a guided Colab orientation for readers who are new to programming, statistics, and machine learning. It uses a small marketing example to connect notebook mechanics with a complete business analytics workflow.


## How to use this notebook

Run the cells from top to bottom. Read the explanation before each code block, inspect every output, and change only one thing at a time while learning. Before sharing your work, use **Runtime > Restart session and run all** in Colab. A notebook that succeeds only because cells were run in a hidden order is not reproducible.

The notebook is safe to run on a CPU. Interactive upload and Google Drive cells are disabled by default so that a full top-to-bottom run does not pause for user input.


## Why this matters (business framing)

Colab is not the business decision. It is the bridge between a question, a data source, executable analysis, visible evidence, and an interpretation that another person can inspect. A well-organized notebook makes assumptions and analytical choices visible instead of hiding them behind a polished chart or model score.


## Agenda

1. Setup and reproducibility
2. Colab as an analytics bridge
3. Notebook anatomy and execution order
4. Naming, saving, and sharing
5. Errors and troubleshooting
6. Data access and file persistence
7. Libraries and environment management
8. A first business analytics workflow
9. Visualization and saved outputs
10. Runtime and accelerator choices
11. Privacy, collaboration, and notebook handoff
12. Exercises


## Learning objectives

After completing this notebook, you should be able to create a reproducible Colab workflow, distinguish the notebook file from its temporary runtime, load and inspect a small dataset, interpret common error messages, fit a simple predictive model, save analysis outputs, select an appropriate runtime, and prepare a notebook for responsible sharing.


## Connection map

Every later companion notebook assumes that you can run cells in order, locate files, inspect data before modeling, save outputs, and restart a clean runtime. Return to this appendix whenever a file path fails, a package disappears, a variable is undefined, or a notebook cannot be rerun from the first cell.


In [ ]:
# ============================================================
# 1. Setup and reproducibility
# ============================================================
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib
import importlib.metadata
import importlib.util
import json
import os
import platform
import random
import sys
import traceback
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

OUT_DIR = Path('appendix_a_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 120)

print(f'Running in Colab: {IN_COLAB}')
print(f'Python: {platform.python_version()}')
print(f'Working directory: {Path.cwd().resolve()}')
print(f'Output directory: {OUT_DIR.resolve()}')


## Utility functions

These helpers keep the practical sections readable. They support compact table display, safe error demonstrations, file checksums, and JSON output.


In [ ]:
# ============================================================
# Utility functions
# ============================================================
def show_table(frame, rows=10):
    """Display a compact copy of a table."""
    display(frame.head(rows).copy())


def sha256_file(path):
    """Return a SHA-256 checksum for a saved file."""
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(65536), b''):
            digest.update(block)
    return digest.hexdigest()


def save_json(payload, path):
    """Save a JSON-serializable object with readable indentation."""
    with open(path, 'w', encoding='utf-8') as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False)


def capture_error(label, function):
    """Run a function and return an error summary without stopping the notebook."""
    try:
        function()
        return {
            'example': label,
            'error_type': 'No error',
            'message': '',
            'first_check': 'No correction needed',
        }
    except Exception as exc:
        first_checks = {
            'NameError': 'Check spelling and confirm that the defining cell ran first.',
            'FileNotFoundError': 'Check the file browser and copy the complete path.',
            'KeyError': 'Inspect the column names before selecting a column.',
            'ModuleNotFoundError': 'Install the package near the top of the notebook.',
            'TypeError': 'Inspect the object type and the operation being requested.',
        }
        return {
            'example': label,
            'error_type': type(exc).__name__,
            'message': str(exc),
            'first_check': first_checks.get(type(exc).__name__, 'Read the final traceback line and inspect the referenced line.'),
        }


## A.1 Colab as a practical analytics bridge

A notebook should make the full analytical chain visible. The business question defines the evidence that is needed. Data and code produce that evidence. Interpretation translates the output into language that can support a decision.


In [ ]:
# A compact map from business question to decision support.
analytics_bridge = pd.DataFrame([
    {'stage': 'Business question', 'example': 'Which customers should receive a retention offer?', 'deliverable': 'Decision contract'},
    {'stage': 'Data', 'example': 'Prior purchases, email activity, and customer status', 'deliverable': 'Documented analysis table'},
    {'stage': 'Code', 'example': 'Cleaning, visualization, and a predictive model', 'deliverable': 'Executable notebook cells'},
    {'stage': 'Evidence', 'example': 'Data checks, charts, and test metrics', 'deliverable': 'Inspectable outputs'},
    {'stage': 'Interpretation', 'example': 'Who is flagged, why, and with what uncertainty?', 'deliverable': 'Decision-ready explanation'},
])
show_table(analytics_bridge)


## A.2 The notebook as an analytics workspace

A Colab notebook contains text cells and code cells, while a runtime performs the actual computation. The notebook file can persist in Drive or on a computer, but variables, temporary uploads, and manually installed packages belong to the runtime and can disappear when the session restarts.


In [ ]:
# ============================================================
# A.2.1 Core notebook elements
# ============================================================
notebook_elements = pd.DataFrame([
    {
        'element': 'Code cell',
        'function': 'Runs Python instructions and displays output below the cell.',
        'beginner_habit': 'Run cells from top to bottom when reproducing an analysis.',
    },
    {
        'element': 'Text cell',
        'function': 'Stores headings, assumptions, methods, and interpretation.',
        'beginner_habit': 'Explain why a step matters, not only what the code does.',
    },
    {
        'element': 'Runtime',
        'function': 'Provides the temporary machine where code executes.',
        'beginner_habit': 'Expect variables, uploads, and manual installs to disappear after restart.',
    },
    {
        'element': 'File browser',
        'function': 'Shows files available to the current runtime and mounted storage.',
        'beginner_habit': 'Check the file location before assuming that Python is wrong.',
    },
])
show_table(notebook_elements)


In [ ]:
# ============================================================
# A.2.2 A first code cell
# ============================================================
visitors = 2500
conversions = 185
conversion_rate = conversions / visitors

print(f'Visitors: {visitors:,}')
print(f'Conversions: {conversions:,}')
print(f'Conversion rate: {conversion_rate:.2%}')


The variables above exist in the current runtime. Restarting the runtime clears them. The code remains in the notebook, so rerunning the cell reconstructs the calculation.


## A.3 Creating, naming, saving, and sharing a notebook

Clear file names reduce confusion during learning and collaboration. A useful name identifies the topic and version, uses an `.ipynb` extension, and avoids vague labels such as `final2` or `new_copy`. Sharing permissions should match the task, and private data, credentials, and unnecessary outputs should be removed before distribution.


In [ ]:
# ============================================================
# A.3.1 Notebook naming audit
# ============================================================
def audit_notebook_name(name):
    lower = name.lower()
    return {
        'name': name,
        'ipynb_extension': lower.endswith('.ipynb'),
        'no_spaces': ' ' not in name,
        'descriptive_topic': any(token in lower for token in ['campaign', 'churn', 'colab', 'analytics']),
        'version_marker': any(token in lower for token in ['_v01', '_v1', '_draft', '_final']),
    }

name_examples = [
    'customer_churn_intro_v01.ipynb',
    'final2.ipynb',
    'Campaign Analysis.ipynb',
    'appendix_a_colab_final.ipynb',
]

name_audit = pd.DataFrame([audit_notebook_name(name) for name in name_examples])
show_table(name_audit)


In [ ]:
# ============================================================
# A.3.2 Sharing checklist
# ============================================================
sharing_checklist = pd.DataFrame([
    {'check': 'Permission matches the collaboration goal', 'example': 'Viewer, commenter, or editor', 'status': 'Review before sharing'},
    {'check': 'Private data removed or access-controlled', 'example': 'Customer records and internal reports', 'status': 'Required'},
    {'check': 'Credentials are not stored in cells', 'example': 'API keys, passwords, and tokens', 'status': 'Required'},
    {'check': 'Notebook runs from top to bottom', 'example': 'Restart session and run all', 'status': 'Required'},
    {'check': 'Outputs are appropriate for the audience', 'example': 'Remove sensitive tables and debugging logs', 'status': 'Review before sharing'},
])
show_table(sharing_checklist)


## A.4 Running Python code and reading error messages

Errors are normal evidence about what Python could not do. Start with the final line of the traceback, identify the error type, and inspect the referenced line. Confirm that earlier cells ran in the intended order before making broad changes.


In [ ]:
# ============================================================
# A.4.1 Common errors, demonstrated safely
# ============================================================
def name_error_demo():
    return undefined_campaign_total + 1


def file_error_demo():
    return pd.read_csv('a_file_that_does_not_exist.csv')


def key_error_demo():
    demo = pd.DataFrame({'spend': [100, 200]})
    return demo['revenue']


error_examples = pd.DataFrame([
    capture_error('Variable was not defined', name_error_demo),
    capture_error('File path was incorrect', file_error_demo),
    capture_error('Column name was incorrect', key_error_demo),
])
show_table(error_examples)


In [ ]:
# ============================================================
# A.4.2 Execution order and state
# ============================================================
analysis_status = 'setup complete'
step_number = 1
print(f'Step {step_number}: {analysis_status}')

step_number += 1
analysis_status = 'data ready'
print(f'Step {step_number}: {analysis_status}')

print('\nA clean rerun should recreate the same sequence without relying on hidden variables.')


## A.5 Bringing data into Colab

The most important distinction is persistence. A direct upload is temporary in the current runtime. A file stored in Google Drive, a database, or another managed system persists beyond the session. Whatever the source, inspect the first rows, dimensions, data types, and missing values immediately after loading.


In [ ]:
# ============================================================
# A.5.1 Create and load a small runtime file
# ============================================================
runtime_sample = pd.DataFrame({
    'campaign_id': ['C101', 'C102', 'C103', 'C104', 'C105'],
    'channel': ['Email', 'Search', 'Social', 'Email', 'Search'],
    'spend': [220, 480, 310, 260, 525],
    'clicks': [42, 65, 51, 48, 70],
    'conversions': [7, 8, 6, 9, 10],
})

runtime_csv = OUT_DIR / 'runtime_campaign_sample.csv'
runtime_sample.to_csv(runtime_csv, index=False)

campaign_sample = pd.read_csv(runtime_csv)
print(f'Loaded {runtime_csv} with shape {campaign_sample.shape}.')
show_table(campaign_sample)


In [ ]:
# ============================================================
# A.5.2 Inspect a newly loaded table before modeling
# ============================================================
data_profile = pd.DataFrame({
    'column': campaign_sample.columns,
    'dtype': campaign_sample.dtypes.astype(str).values,
    'missing_values': campaign_sample.isna().sum().values,
    'unique_values': campaign_sample.nunique(dropna=False).values,
})

print(f'Rows: {campaign_sample.shape[0]:,}')
print(f'Columns: {campaign_sample.shape[1]:,}')
show_table(data_profile)


In [ ]:
# ============================================================
# A.5.3 Optional Colab upload and Google Drive patterns
# These switches remain False so that Run all never pauses.
# ============================================================
RUN_INTERACTIVE_UPLOAD = False
MOUNT_GOOGLE_DRIVE = False

if RUN_INTERACTIVE_UPLOAD:
    if not IN_COLAB:
        print('Interactive upload is available only inside Google Colab.')
    else:
        from google.colab import files
        uploaded = files.upload()
        first_file = next(iter(uploaded))
        uploaded_df = pd.read_csv(first_file)
        print(f'Uploaded and loaded: {first_file}')
        display(uploaded_df.head())
else:
    print('Interactive upload skipped. Set RUN_INTERACTIVE_UPLOAD = True in Colab when needed.')

if MOUNT_GOOGLE_DRIVE:
    if not IN_COLAB:
        print('Google Drive mounting is available only inside Google Colab.')
    else:
        from google.colab import drive
        drive.mount('/content/drive')
        print('Drive mounted. Replace the path below with your own file path.')
        # df_drive = pd.read_csv('/content/drive/MyDrive/datasets/campaigns.csv')
else:
    print('Google Drive mount skipped. Set MOUNT_GOOGLE_DRIVE = True in Colab when needed.')


A public URL can be passed directly to `pandas.read_csv()` when the source is stable and truly public. Database and API access require project-approved authentication and privacy controls. Do not place passwords or tokens directly in a notebook cell.


## A.6 Installing libraries and managing a reproducible environment

Colab includes many common analytics libraries, but the runtime is temporary. Place required installation commands near the top of the notebook and record package versions when compatibility matters. Use a fixed random seed for demonstrations, while remembering that a seed improves reproducibility but does not validate a model.


In [ ]:
# ============================================================
# A.6.1 Environment and package versions
# ============================================================
def package_version(distribution_name):
    try:
        return importlib.metadata.version(distribution_name)
    except importlib.metadata.PackageNotFoundError:
        return 'not installed'

version_table = pd.DataFrame([
    {'component': 'Python', 'version': platform.python_version()},
    {'component': 'NumPy', 'version': package_version('numpy')},
    {'component': 'pandas', 'version': package_version('pandas')},
    {'component': 'Matplotlib', 'version': package_version('matplotlib')},
    {'component': 'scikit-learn', 'version': package_version('scikit-learn')},
])
show_table(version_table)

print('Example installation command for a required package:')
print('%pip install package_name==version_number')


In [ ]:
# ============================================================
# A.6.2 Random seeds make demonstrations repeatable
# ============================================================
def repeatable_draw(seed, size=5):
    rng = np.random.default_rng(seed)
    return rng.normal(size=size)

first_draw = repeatable_draw(SEED)
second_draw = repeatable_draw(SEED)

print('First draw: ', np.round(first_draw, 4))
print('Second draw:', np.round(second_draw, 4))
print('Identical:', np.allclose(first_draw, second_draw))


## A.7 A first business analytics workflow in Colab

The purpose of this example is to show notebook rhythm, not to build a production model. We begin with a decision contract, create a small simulated marketing dataset, check the data, visualize a relationship, fit a simple logistic regression model, and interpret the result cautiously.


In [ ]:
# ============================================================
# A.7.1 Decision contract
# ============================================================
decision_contract = pd.DataFrame([
    {'field': 'Unit of analysis', 'definition': 'One customer-campaign exposure'},
    {'field': 'Prediction target', 'definition': 'Whether the customer converts'},
    {'field': 'Available information', 'definition': 'Spend, discount, email opens, prior purchases, and channel'},
    {'field': 'Candidate action', 'definition': 'Prioritize follow-up for high-probability opportunities'},
    {'field': 'Primary caution', 'definition': 'Prediction does not establish that a discount causes conversion'},
])
show_table(decision_contract)


In [ ]:
# ============================================================
# A.7.2 Generate a synthetic marketing dataset
# ============================================================
rng = np.random.default_rng(SEED)
n = 600

marketing = pd.DataFrame({
    'customer_id': [f'U{i:04d}' for i in range(n)],
    'spend': rng.normal(500, 120, n).clip(100, 900),
    'discount': rng.choice([0, 5, 10, 15], size=n, p=[0.25, 0.30, 0.30, 0.15]),
    'email_opens': rng.poisson(2.2, n),
    'prior_purchases': rng.poisson(1.6, n),
    'channel': rng.choice(['Email', 'Search', 'Social'], size=n, p=[0.45, 0.30, 0.25]),
})

channel_effect = marketing['channel'].map({'Email': 0.20, 'Search': 0.35, 'Social': -0.10})
logit = (
    -3.25
    + 0.0032 * marketing['spend']
    + 0.075 * marketing['discount']
    + 0.23 * marketing['email_opens']
    + 0.18 * marketing['prior_purchases']
    + channel_effect
)
probability = 1 / (1 + np.exp(-logit))
marketing['converted'] = rng.binomial(1, probability)

print(f'Dataset shape: {marketing.shape}')
print(f'Observed conversion rate: {marketing["converted"].mean():.2%}')
show_table(marketing)


In [ ]:
# ============================================================
# A.7.3 Sanity checks and a first visualization
# ============================================================
quality_checks = pd.DataFrame([
    {'check': 'Duplicate customer IDs', 'value': int(marketing['customer_id'].duplicated().sum()), 'expected': 0},
    {'check': 'Missing cells', 'value': int(marketing.isna().sum().sum()), 'expected': 0},
    {'check': 'Minimum spend', 'value': round(float(marketing['spend'].min()), 2), 'expected': 'at least 100'},
    {'check': 'Maximum discount', 'value': int(marketing['discount'].max()), 'expected': 15},
    {'check': 'Target classes', 'value': int(marketing['converted'].nunique()), 'expected': 2},
])
show_table(quality_checks)

conversion_by_discount = (
    marketing.groupby('discount', as_index=False)
    .agg(conversion_rate=('converted', 'mean'), observations=('converted', 'size'))
)

plt.figure(figsize=(7, 4))
plt.plot(conversion_by_discount['discount'], conversion_by_discount['conversion_rate'], marker='o')
plt.xlabel('Discount (%)')
plt.ylabel('Observed conversion rate')
plt.title('Conversion by discount level in the simulated data')
plt.ylim(0, min(1, conversion_by_discount['conversion_rate'].max() + 0.15))
plt.grid(alpha=0.25)
plt.show()

show_table(conversion_by_discount)


In [ ]:
# ============================================================
# A.7.4 Train and evaluate a simple logistic regression model
# ============================================================
feature_columns = ['spend', 'discount', 'email_opens', 'prior_purchases', 'channel']
numeric_features = ['spend', 'discount', 'email_opens', 'prior_purchases']
categorical_features = ['channel']

X = marketing[feature_columns]
y = marketing['converted']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=SEED,
    stratify=y,
)

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocess = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features),
])

model = Pipeline([
    ('preprocess', preprocess),
    ('classifier', LogisticRegression(max_iter=1000, random_state=SEED)),
])

model.fit(X_train, y_train)
test_probability = model.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= 0.50).astype(int)

metrics = {
    'accuracy': accuracy_score(y_test, test_prediction),
    'balanced_accuracy': balanced_accuracy_score(y_test, test_prediction),
    'roc_auc': roc_auc_score(y_test, test_probability),
}

metrics_table = pd.DataFrame([
    {'metric': name, 'value': value}
    for name, value in metrics.items()
])
show_table(metrics_table)

cm = confusion_matrix(y_test, test_prediction)
confusion_table = pd.DataFrame(
    cm,
    index=['Actual 0', 'Actual 1'],
    columns=['Predicted 0', 'Predicted 1'],
)
show_table(confusion_table)


In [ ]:
# ============================================================
# A.7.5 Inspect model directions without making causal claims
# ============================================================
transformed_names = model.named_steps['preprocess'].get_feature_names_out()
coefficients = model.named_steps['classifier'].coef_[0]

coefficient_table = pd.DataFrame({
    'model_feature': transformed_names,
    'coefficient': coefficients,
})
coefficient_table['absolute_coefficient'] = coefficient_table['coefficient'].abs()
coefficient_table = coefficient_table.sort_values('absolute_coefficient', ascending=False)

show_table(coefficient_table[['model_feature', 'coefficient']], rows=20)
print('Interpretation reminder: coefficient direction describes model association after preprocessing. It does not prove causality.')


## A.8 Visualization and output as communication

A chart inside a notebook is useful for exploration, but a business workflow often needs saved files for a report, presentation, audit, or handoff. Save the figure and the data used to create it. A second analyst should not have to reverse-engineer a chart from pixels.


In [ ]:
# ============================================================
# A.8.1 Save a chart and analysis tables
# ============================================================
figure_path = OUT_DIR / 'conversion_by_discount.png'
data_path = OUT_DIR / 'synthetic_marketing_data.csv'
summary_path = OUT_DIR / 'conversion_by_discount.csv'
prediction_path = OUT_DIR / 'test_predictions.csv'
metrics_path = OUT_DIR / 'model_metrics.json'

plt.figure(figsize=(7, 4))
plt.plot(conversion_by_discount['discount'], conversion_by_discount['conversion_rate'], marker='o')
plt.xlabel('Discount (%)')
plt.ylabel('Observed conversion rate')
plt.title('Conversion by discount level')
plt.grid(alpha=0.25)
plt.savefig(figure_path, dpi=150, bbox_inches='tight')
plt.close()

marketing.to_csv(data_path, index=False)
conversion_by_discount.to_csv(summary_path, index=False)

predictions = X_test.copy()
predictions['actual_converted'] = y_test.to_numpy()
predictions['predicted_probability'] = test_probability
predictions['predicted_class_050'] = test_prediction
predictions.to_csv(prediction_path, index=False)

save_json({name: float(value) for name, value in metrics.items()}, metrics_path)

print('Saved analysis artifacts:')
for path in [figure_path, data_path, summary_path, prediction_path, metrics_path]:
    print(f'  {path}')


In [ ]:
# ============================================================
# A.8.2 Build an output manifest
# ============================================================
artifact_paths = [figure_path, data_path, summary_path, prediction_path, metrics_path]
artifact_manifest = pd.DataFrame([
    {
        'file': path.name,
        'size_bytes': path.stat().st_size,
        'sha256_prefix': sha256_file(path)[:12],
    }
    for path in artifact_paths
])
show_table(artifact_manifest)

DOWNLOAD_OUTPUTS = False
if DOWNLOAD_OUTPUTS and IN_COLAB:
    from google.colab import files
    for path in artifact_paths:
        files.download(str(path))
elif DOWNLOAD_OUTPUTS:
    print('Automatic download is available only inside Google Colab.')
else:
    print('Downloads skipped. Set DOWNLOAD_OUTPUTS = True in Colab when needed.')


## A.9 Accelerators, limits, and responsible resource use

Choose a runtime for the workload, not for prestige. Data inspection, charts, linear models, and many tree-based methods usually work well on a CPU. Neural networks for images, text, or large data may benefit from a GPU, but selecting a GPU does not guarantee that the code uses it.


In [ ]:
# ============================================================
# A.9.1 Inspect the current runtime
# ============================================================
runtime_rows = [
    {'item': 'Environment', 'value': 'Google Colab' if IN_COLAB else 'Local or hosted Jupyter'},
    {'item': 'Python version', 'value': platform.python_version()},
    {'item': 'Operating system', 'value': platform.platform()},
    {'item': 'Logical CPU count', 'value': os.cpu_count()},
    {'item': 'Working directory', 'value': str(Path.cwd().resolve())},
]

if importlib.util.find_spec('torch') is not None:
    import torch
    runtime_rows.extend([
        {'item': 'PyTorch version', 'value': torch.__version__},
        {'item': 'CUDA available to PyTorch', 'value': bool(torch.cuda.is_available())},
        {'item': 'Selected PyTorch device', 'value': 'cuda' if torch.cuda.is_available() else 'cpu'},
    ])

runtime_report = pd.DataFrame(runtime_rows)
show_table(runtime_report, rows=20)


In [ ]:
# ============================================================
# A.9.2 Task-centered runtime guide
# ============================================================
runtime_guide = pd.DataFrame([
    {
        'runtime': 'CPU',
        'best_fit': 'Pandas, visualization, regression, clustering, and many classical models',
        'important_limit': 'Can be slow for large neural networks or high-resolution images',
    },
    {
        'runtime': 'GPU',
        'best_fit': 'Compatible deep learning, image, text, and matrix-heavy workloads',
        'important_limit': 'Availability is limited and the code must move computation to the GPU',
    },
    {
        'runtime': 'TPU',
        'best_fit': 'Selected TensorFlow or JAX workloads designed for TPU execution',
        'important_limit': 'Requires task-specific setup and is unnecessary for most beginner work',
    },
    {
        'runtime': 'Paid or enterprise',
        'best_fit': 'Longer runs, additional memory, team governance, or dedicated resources',
        'important_limit': 'Costs money and still requires efficient, documented code',
    },
])
show_table(runtime_guide)


## A.10 Collaboration, privacy, and troubleshooting

A notebook is both an analytical document and a potential data leak. Share the minimum necessary information, use the minimum necessary permission, and separate credentials from code. Many apparent Colab failures are workflow failures caused by lost runtime state, an incorrect path, a missing package, or cells executed out of order.


In [ ]:
# ============================================================
# A.10.1 Privacy and secret-handling patterns
# ============================================================
def privacy_column_audit(columns):
    high_risk_terms = ['email', 'phone', 'address', 'ssn', 'password', 'token', 'secret', 'api_key']
    identifier_terms = ['customer_id', 'user_id', 'account_id', 'device_id']
    rows = []
    for column in columns:
        lower = column.lower()
        if any(term in lower for term in high_risk_terms):
            category = 'Sensitive or credential-like'
            action = 'Remove, mask, or use approved secure handling'
        elif any(term in lower for term in identifier_terms):
            category = 'Identifier'
            action = 'Confirm that row-level identity is necessary'
        else:
            category = 'No keyword flag'
            action = 'Still review values and business context'
        rows.append({'column': column, 'risk_flag': category, 'recommended_action': action})
    return pd.DataFrame(rows)

privacy_audit = privacy_column_audit(marketing.columns)
show_table(privacy_audit, rows=20)


def get_project_secret(name):
    """Read a secret without hard-coding it in the notebook."""
    if IN_COLAB:
        from google.colab import userdata
        return userdata.get(name)
    return os.getenv(name)

print('Safe pattern prepared. The notebook does not request or print any real secret.')
print('Example usage: api_key = get_project_secret("PROJECT_API_KEY")')


In [ ]:
# ============================================================
# A.10.2 Troubleshooting guide and environment diagnostic
# ============================================================
troubleshooting_guide = pd.DataFrame([
    {'symptom': 'FileNotFoundError', 'practical_fix': 'Check the file browser, mount Drive if needed, and copy the complete path.', 'long_term_habit': 'Keep data paths in one labeled section.'},
    {'symptom': 'NameError', 'practical_fix': 'Run setup and data cells again, or restart and run all.', 'long_term_habit': 'Design notebooks to run from top to bottom.'},
    {'symptom': 'ModuleNotFoundError', 'practical_fix': 'Install the package and import it again.', 'long_term_habit': 'Place installation commands near the top.'},
    {'symptom': 'Different results after rerunning', 'practical_fix': 'Set seeds and rerun from a clean runtime.', 'long_term_habit': 'Record data versions and random states.'},
    {'symptom': 'Notebook is not saving', 'practical_fix': 'Close duplicate tabs and save a copy in Drive.', 'long_term_habit': 'Work in one active tab per notebook.'},
])
show_table(troubleshooting_guide)

required_files = [runtime_csv, figure_path, data_path, metrics_path]
required_packages = ['numpy', 'pandas', 'matplotlib', 'sklearn']

diagnostic_rows = []
for path in required_files:
    diagnostic_rows.append({
        'diagnostic': f'File: {path}',
        'status': 'available' if path.exists() else 'missing',
    })
for package in required_packages:
    diagnostic_rows.append({
        'diagnostic': f'Package: {package}',
        'status': 'available' if importlib.util.find_spec(package) is not None else 'missing',
    })

diagnostic_report = pd.DataFrame(diagnostic_rows)
show_table(diagnostic_report, rows=20)


In [ ]:
# ============================================================
# A.10.3 A compact clean-run reproducibility test
# ============================================================
def compact_workflow(seed):
    local_rng = np.random.default_rng(seed)
    rows = 240
    data = pd.DataFrame({
        'spend': local_rng.normal(500, 100, rows),
        'opens': local_rng.poisson(2, rows),
    })
    local_logit = -2.6 + 0.003 * data['spend'] + 0.30 * data['opens']
    data['converted'] = local_rng.binomial(1, 1 / (1 + np.exp(-local_logit)))

    train, test = train_test_split(
        data,
        test_size=0.25,
        random_state=seed,
        stratify=data['converted'],
    )
    local_model = LogisticRegression(max_iter=1000, random_state=seed)
    local_model.fit(train[['spend', 'opens']], train['converted'])
    probability = local_model.predict_proba(test[['spend', 'opens']])[:, 1]
    return {
        'rows': rows,
        'test_roc_auc': round(float(roc_auc_score(test['converted'], probability)), 10),
        'prediction_checksum': hashlib.sha256(np.asarray(probability).tobytes()).hexdigest()[:16],
    }

clean_run_1 = compact_workflow(SEED)
clean_run_2 = compact_workflow(SEED)

reproducibility_check = pd.DataFrame([
    {'run': 'Run 1', **clean_run_1},
    {'run': 'Run 2', **clean_run_2},
])
show_table(reproducibility_check)
print('Reproducible:', clean_run_1 == clean_run_2)


## A.11 From tool use to analytical habit

The long-term goal is not simply to operate Colab. It is to build a repeatable habit: state the question, make the data path visible, run code in a reproducible order, inspect output critically, communicate findings clearly, and protect information responsibly.


In [ ]:
# ============================================================
# A.11.1 Notebook handoff card and readiness checklist
# ============================================================
handoff_card = {
    'notebook_name': 'Appendix_A_Google_Colab.ipynb',
    'purpose': 'Beginner orientation to a reproducible Colab business analytics workflow',
    'unit_of_analysis': 'One synthetic customer-campaign exposure',
    'target': 'converted',
    'data_source': 'Synthetic data generated inside the notebook',
    'random_seed': SEED,
    'primary_model': 'Logistic regression',
    'primary_metrics': list(metrics.keys()),
    'output_directory': str(OUT_DIR),
    'known_limitations': [
        'Synthetic data does not represent a real company or market.',
        'The model is instructional and not approved for deployment.',
        'Associations and coefficients should not be interpreted as causal effects.',
    ],
    'created_utc': datetime.now(timezone.utc).isoformat(),
}

handoff_path = OUT_DIR / 'notebook_handoff_card.json'
save_json(handoff_card, handoff_path)

readiness_checklist = pd.DataFrame([
    {'item': 'Business question and unit of analysis are stated', 'status': 'complete'},
    {'item': 'Data source and persistence are understood', 'status': 'complete'},
    {'item': 'Data checks occur before modeling', 'status': 'complete'},
    {'item': 'Random seed and package versions are visible', 'status': 'complete'},
    {'item': 'Model is evaluated on held-out data', 'status': 'complete'},
    {'item': 'Charts and supporting data are saved', 'status': 'complete'},
    {'item': 'Credentials are not stored in code', 'status': 'complete'},
    {'item': 'Notebook can be restarted and run from top to bottom', 'status': 'verify in Colab before sharing'},
])

show_table(readiness_checklist, rows=20)
print(f'Handoff card saved to: {handoff_path}')


## Decision guide

Use a direct runtime upload for a small, temporary classroom file. Use Google Drive or another managed source when the file must persist or be shared. Keep installation commands and imports near the top, inspect every dataset immediately after loading, and restart the runtime when hidden state makes the workflow confusing. Use a CPU unless the workload has a credible need for an accelerator. Before sharing, remove sensitive outputs, confirm permissions, and rerun the notebook from a clean session.


## Exercises

1. Change the visitor and conversion counts in the first calculation. Confirm that the percentage formatting still works.

2. Add three notebook names to `name_examples`. Explain why each name passes or fails the audit.

3. Introduce one missing value into `campaign_sample`. Update the data profile and identify the affected column.

4. In Colab, upload a small CSV by setting `RUN_INTERACTIVE_UPLOAD = True`. Inspect its first rows, dimensions, types, and missing values.

5. Change the synthetic marketing seed and rerun the workflow. Which outputs change, and which notebook structures should remain unchanged?

6. Change the classification threshold from 0.50 to 0.35. Recalculate the confusion matrix and explain the business tradeoff.

7. Create a second chart that compares conversion across channels. Save both the chart and its summary table.

8. Add a fake `customer_email` column to a copy of the dataset and rerun the privacy audit. Remove the column before saving a shareable file.

9. Restart the Colab runtime and run all cells. Record the first cell that fails, if any, and correct the workflow rather than manually recreating hidden state.

10. Extend the handoff card with the notebook owner, review date, and intended audience.


In [ ]:
# Optional exercise starter: compare two probability thresholds.
exercise_thresholds = [0.35, 0.50]
exercise_rows = []

for threshold in exercise_thresholds:
    prediction = (test_probability >= threshold).astype(int)
    matrix = confusion_matrix(y_test, prediction)
    tn, fp, fn, tp = matrix.ravel()
    exercise_rows.append({
        'threshold': threshold,
        'true_negatives': int(tn),
        'false_positives': int(fp),
        'false_negatives': int(fn),
        'true_positives': int(tp),
        'accuracy': accuracy_score(y_test, prediction),
    })

exercise_threshold_table = pd.DataFrame(exercise_rows)
show_table(exercise_threshold_table)
